In [150]:
import openvsp as vsp
import json
import pandas as pd
import numpy as np

## Objective: 
Given an input wing and tail geometry file (from `surface_defn.ipynb`), this script interfaces with the OpenVSP API and models the wing, hstab, and vstab as defined. 

We further run a CompGeom analysis through the API, and access the information to calculate weights for the wing, hstab, and vstab using statistical equations from Raymer. These weights are then assigned in OpenVSP, assuming a uniform density. 

### Geometry Builder

In [151]:
def auto_wing(global_x_transl, fuse, wing_foil, tail_foil, geom_def, filename):

    #Read JSON file
    with open(f"{geom_def}", "r") as file:
        geom = json.load(file)

    wing = geom["wing"]
    hstab = geom["hstab"]
    vstab = geom["vstab"]

    vsp.ClearVSPModel()


    #Insert the fuselage
    vsp.InsertVSPFile(fuse, "")


    #Define the Wing
    wing_id = vsp.AddGeom("WING", "")
    vsp.SetGeomName(wing_id, "Main_Wing")

    vsp.SetParmVal(wing_id, "Span", "XSec_1", wing["b_w"]/2)
    vsp.SetParmVal(wing_id, "Root_Chord", "XSec_1", wing["c_r_w"])
    vsp.SetParmVal(wing_id, "Tip_Chord", "XSec_1", wing["c_t_w"])
    vsp.SetParmVal(wing_id, "Sweep_Location", "XSec_1", 0)
    vsp.SetParmVal(wing_id, "Sweep", "XSec_1", wing["swp_w"])
    vsp.SetParmVal(wing_id, "X_Rel_Location", "XForm", global_x_transl)

    #Assign to set0
    set_0_idx = vsp.GetSetIndex("Set_0")
    vsp.vsp.SetSetFlag(wing_id, set_0_idx, True)

    #Set Wing Airfoil
    xsec_surf = vsp.GetXSecSurf(wing_id, 0)
    vsp.ChangeXSecShape(xsec_surf, 0, vsp.XS_FILE_AIRFOIL)
    vsp.ChangeXSecShape(xsec_surf, 1, vsp.XS_FILE_AIRFOIL)

    root_xsec = vsp.GetXSec(xsec_surf, 0)
    tip_xsec = vsp.GetXSec(xsec_surf, 1)

    vsp.ReadFileAirfoil(root_xsec, wing_foil)
    vsp.ReadFileAirfoil(tip_xsec, wing_foil)

    #Define the Flaps
    flap_id = vsp.AddSubSurf(wing_id, vsp.SS_CONTROL)
    vsp.SetSubSurfName(wing_id, flap_id, "Flaps")
    flap_parm = vsp.GetSubSurfParmIDs(flap_id)

    for parm_id in flap_parm:
        n_parm = vsp.GetParmName(parm_id)

        if n_parm == "EtaFlag": 
            vsp.SetParmVal(parm_id, 1.0)
        #elif n_parm == "SE_Const_Flag":
        #    vsp.SetParmVal(parm_id, 0.0)
        elif n_parm == "EtaStart":
            vsp.SetParmVal(parm_id, wing["flap_1_span"])
        elif n_parm == "EtaEnd":
            vsp.SetParmVal(parm_id, wing["flap_2_span"])
        elif n_parm == "Length_C_Start":
            vsp.SetParmVal(parm_id, wing["flap_c_frac1"])
        elif n_parm == "Length_C_End":
            vsp.SetParmVal(parm_id, wing["flap_c_frac2"])

    #Define Ailerons
    ail_id = vsp.AddSubSurf(wing_id, vsp.SS_CONTROL)
    vsp.SetSubSurfName(wing_id, ail_id, "Ailerons")
    ail_parm = vsp.GetSubSurfParmIDs(ail_id)

    for parm_id in ail_parm:
        n_parm = vsp.GetParmName(parm_id)

        if n_parm == "EtaFlag": 
            vsp.SetParmVal(parm_id, 1.0)
        #elif n_parm == "SE_Const_Flag":
        #    vsp.SetParmVal(parm_id, 0.0)
        elif n_parm == "EtaStart":
            vsp.SetParmVal(parm_id, wing["ail_1_span"])
        elif n_parm == "EtaEnd":
            vsp.SetParmVal(parm_id, wing["ail_2_span"])
        elif n_parm == "Length_C_Start":
            vsp.SetParmVal(parm_id, wing["ail_c_frac1"])
        elif n_parm == "Length_C_End":
            vsp.SetParmVal(parm_id, wing["ail_c_frac2"])

    #Define Slats
    slat_id = vsp.AddSubSurf(wing_id, vsp.SS_CONTROL)
    vsp.SetSubSurfName(wing_id, slat_id, "Slats")
    slat_parm = vsp.GetSubSurfParmIDs(slat_id)

    for parm_id in slat_parm:
        n_parm = vsp.GetParmName(parm_id)

        if n_parm == "EtaFlag": 
            vsp.SetParmVal(parm_id, 1.0)
        elif n_parm == "SE_Const_Flag":
            vsp.SetParmVal(parm_id, 0.0)
        elif n_parm == "LE_Flag":
            vsp.SetParmVal(parm_id, 1.0)
        elif n_parm == "EtaStart":
            vsp.SetParmVal(parm_id, wing["slat_1_span"])
        elif n_parm == "EtaEnd":
            vsp.SetParmVal(parm_id, wing["slat_2_span"])
        elif n_parm == "Length_C_Start":
            vsp.SetParmVal(parm_id, wing["slat_c_frac1"])
        elif n_parm == "Length_C_End":
            vsp.SetParmVal(parm_id, wing["slat_c_frac2"])


    #Define the HStab
    hstab_id = vsp.AddGeom("WING", "")
    vsp.SetGeomName(hstab_id, "HStab")
    vsp.SetSetFlag(hstab_id, set_0_idx, True)

    vsp.SetParmVal(hstab_id, "Span", "XSec_1", hstab["b_HT"]/2)
    vsp.SetParmVal(hstab_id, "Root_Chord", "XSec_1", hstab["c_r_HT"])
    vsp.SetParmVal(hstab_id, "Tip_Chord", "XSec_1", hstab["c_t_HT"])
    vsp.SetParmVal(hstab_id, "Sweep_Location", "XSec_1", 0)
    vsp.SetParmVal(hstab_id, "Sweep", "XSec_1", hstab["swp_HT"])
    vsp.SetParmVal(hstab_id, "X_Rel_Location", "XForm", hstab["x_loc_HT"] + global_x_transl)
    vsp.SetParmVal(hstab_id, "Y_Rel_Location", "XForm", hstab["Y_loc"])
    vsp.SetParmVal(hstab_id, "Z_Rel_Location", "XForm", hstab["Z_loc"])

    #Define the HStab Airfoil
    hstab_xsec_surf = vsp.GetXSecSurf(hstab_id, 0)
    vsp.ChangeXSecShape(hstab_xsec_surf, 0, vsp.XS_FILE_AIRFOIL)
    vsp.ChangeXSecShape(hstab_xsec_surf, 1, vsp.XS_FILE_AIRFOIL)

    hstab_root_xsec = vsp.GetXSec(hstab_xsec_surf, 0)
    hstab_tip_xsec = vsp.GetXSec(hstab_xsec_surf, 1)

    vsp.ReadFileAirfoil(hstab_root_xsec, tail_foil)
    vsp.ReadFileAirfoil(hstab_tip_xsec, tail_foil)


    #Define the Vstab
    vstab_id = vsp.AddGeom("WING", "")
    vsp.SetGeomName(vstab_id, "VStab")
    vsp.SetSetFlag(vstab_id, set_0_idx, True)

    vsp.SetParmVal(vstab_id, "Span", "XSec_1", vstab["b_VT"]/2)
    vsp.SetParmVal(vstab_id, "Root_Chord", "XSec_1", vstab["c_r_VT"])
    vsp.SetParmVal(vstab_id, "Tip_Chord", "XSec_1", vstab["c_t_VT"])
    vsp.SetParmVal(vstab_id, "Sweep_Location", "XSec_1", 0)
    vsp.SetParmVal(vstab_id, "Sweep", "XSec_1", vstab["swp_VT"])
    vsp.SetParmVal(vstab_id, "X_Rel_Location", "XForm", vstab["x_loc_VT"] + global_x_transl)
    vsp.SetParmVal(vstab_id, "Y_Rel_Location", "XForm", vstab["Y_loc"])
    vsp.SetParmVal(vstab_id, "Z_Rel_Location", "XForm", vstab["Z_loc"])
    vsp.SetParmVal(vstab_id, "X_Rel_Rotation", "XForm", vstab["X_rot"])

    #Define the VStab Airfoil
    vstab_xsec_surf = vsp.GetXSecSurf(vstab_id, 0)
    vsp.ChangeXSecShape(vstab_xsec_surf, 0, vsp.XS_FILE_AIRFOIL)
    vsp.ChangeXSecShape(vstab_xsec_surf, 1, vsp.XS_FILE_AIRFOIL)

    vstab_root_xsec = vsp.GetXSec(vstab_xsec_surf, 0)
    vstab_tip_xsec = vsp.GetXSec(vstab_xsec_surf, 1)

    vsp.ReadFileAirfoil(vstab_root_xsec, tail_foil)
    vsp.ReadFileAirfoil(vstab_tip_xsec, tail_foil)

    #Define the Rudder
    rudder_id = vsp.AddSubSurf(vstab_id, vsp.SS_CONTROL)
    vsp.SetSubSurfName(vstab_id, rudder_id, "Rudder")
    rud_parm = vsp.GetSubSurfParmIDs(rudder_id)

    for parm_id in rud_parm:
        n_parm = vsp.GetParmName(parm_id)

        if n_parm == "EtaFlag":
            vsp.SetParmVal(parm_id, 1.0)
        elif n_parm == "EtaStart":
            vsp.SetParmVal(parm_id, vstab["rud_1_span"])
        elif n_parm == "EtaEnd":
            vsp.SetParmVal(parm_id, vstab["rud_2_span"])
        elif n_parm == "Length_C_Start":
            vsp.SetParmVal(parm_id, vstab["rud_c_frac"])
        elif n_parm == "Length_C_End":
            vsp.SetParmVal(parm_id, vstab["rud_c_frac"])


    vsp.Update()
    vsp.WriteVSPFile(filename)


wing_foil = "/Users/ryuyaiwase/Desktop/Airfoil Libby/NASA SC(2)-0406.dat"
tail_foil = "/Users/ryuyaiwase/Desktop/Airfoil Libby/NACA_16-006.dat"
fuselage = "/Users/ryuyaiwase/Desktop/OpenVSP-3.47.0-MacOS/VSP Files/SIMPLE_F24HH_FUSE.vsp3"

vsp_filename = "F24HH"

auto_wing(global_x_transl=0, fuse=fuselage, wing_foil=wing_foil, tail_foil=tail_foil, geom_def="airplane_geom.json", filename=f"{vsp_filename}.vsp3")

### Run CompGeom

In [152]:
def compgeom(plane):
    vsp.ClearVSPModel()

    #Insert previously defined aircraft
    vsp.ReadVSPFile(plane)

    #Run CompGeom
    print(f"Running CompGeom on {plane}...")
    vsp.ComputeCompGeom(vsp.SET_ALL, False, 1) # Set last to 1 to get text printout, 0 for none

    #Access Results
    compgeom_res_id = vsp.FindLatestResultsID("Comp_Geom")
    dat_names = vsp.GetAllDataNames(compgeom_res_id)

    vsp.Update()
    vsp.WriteVSPFile(f"{vsp_filename}.vsp3")

plane = f"{vsp_filename}.vsp3"
compgeom(plane=plane)

Running CompGeom on F24HH.vsp3...


### Parse CompGeom output txt file to find control surface areas
!! ENSURE CORRECT COMPGEOM FILE INPUT !!

In [153]:
def cntrl_sfs_areas(compgeom_txt):
    with open(compgeom_txt, 'r') as file: 
        lines = file.readlines()
    
    for i, line in enumerate(lines):
        if "SS_Theo_Area" in line:
            start_idx = i
            break

    ss_area_df = pd.read_csv(compgeom_txt, skiprows=start_idx, sep='\s+', on_bad_lines='skip')
    display(ss_area_df)

    ss_areas = {
        "Wing_Flap_Area": ss_area_df.at[0, "SS_Theo_Area"],
        "Wing_Aileron_Area": ss_area_df.at[1, "SS_Theo_Area"],
        "Wing_Slat_Area": ss_area_df.at[2, "SS_Theo_Area"],
        "VStab_Rudder_Area": ss_area_df.at[3, "SS_Theo_Area"]
    }

    display(ss_areas)

    return ss_areas


compgeom_file = f"{vsp_filename}_CompGeom.txt"
ss_areas = cntrl_sfs_areas(compgeom_file)

,SS_Theo_Area,SS_Wet_Area,SS_Name
0,106.829,106.829,"Main_Wing,Flaps"
1,40.422,40.422,"Main_Wing,Ailerons"
2,104.179,104.179,"Main_Wing,Slats"
3,53.331,53.331,"VStab,Rudder"


{'Wing_Flap_Area': np.float64(106.829),
 'Wing_Aileron_Area': np.float64(40.422),
 'Wing_Slat_Area': np.float64(104.179),
 'VStab_Rudder_Area': np.float64(53.331)}

### Parse CompGeom output txt file to find wing and tail volumes
!! ENSURE CORRECT COMPGEOM FILE INPUT !!

In [154]:
def sfs_vol(compgeom_txt):
    with open(compgeom_txt, 'r') as file: 
        lines = file.readlines()
    
    for i, line in enumerate(lines):
        if "Main_Wing" in line:
            start_idx = i
            break

    s_vol_df = pd.read_csv(compgeom_txt, skiprows=5, sep='\s+', on_bad_lines='skip', 
                           usecols=['Theo_Area', 'Wet_Area', 'Theo_Vol', 'Wet_Vol', 'Name'])
    #display(s_vol_df)

    des_name = ['Main_Wing', 'Main_Wing', 'HStab', 'VStab', 'VStab']
    s_filt = s_vol_df[s_vol_df['Name'].isin(des_name)]
    print(s_filt)

    s_vols = {
        "Wing_Vol_tot": s_filt.loc[s_filt['Name'] == 'Main_Wing', 'Theo_Vol'].astype(float).sum(),
        "Hstab_Vol_tot": s_filt.loc[s_filt['Name'] == 'HStab', 'Theo_Vol'].astype(float).iloc[0],
        "VStab_Vol_tot": s_filt.loc[s_filt['Name'] == 'VStab', 'Theo_Vol'].astype(float).sum()
    }

    display(s_vols)

    return s_vols


compgeom_file = f"{vsp_filename}_CompGeom.txt"
surf_vols = sfs_vol(compgeom_file)

   Theo_Area Wet_Area Theo_Vol  Wet_Vol       Name
5    471.015  455.328  130.248  129.586  Main_Wing
6    471.015  455.328  130.248  129.586  Main_Wing
7    153.200  146.154   24.822   24.372      HStab
8    153.200  146.154   24.822   24.372      HStab
9    107.533  102.067   14.823   14.481      VStab
10   107.533  102.067   14.823   14.481      VStab


{'Wing_Vol_tot': np.float64(260.496),
 'Hstab_Vol_tot': np.float64(24.822),
 'VStab_Vol_tot': np.float64(29.646)}

### Weights - Wing
Raymer Eq. 15.1 gives statistical equations to estimate the weight of the wing. 
$\begin{equation}
W_{wing} = 0.0103K_{dw}K_{vs}(W_{dg}N_z)^{0.5}S_{w}^{0.622}A^{0.785}(t/c)_{root}\times (1+\lambda)^{0.05}(\text{cos}\Lambda)^{-1.0}S_{csw}^{0.04}
\end{equation}$

Where:
- $K_{dw} = 1.0$
- $K_{vs} = 1.0$
- $W_{dg}$: Flight Design Gross Weight (typically 50-60% internal fuel)
- $N_z = 7 * 1.5 = 10.5$
- $S_w$: Trapezoidal Wing Area (incl. fuse area)
- $A$: Aspect Ratio
- $(t/c)_{root}$: Airfoil thickness at root
- $\lambda$: Taper Ratio
- $\Lambda$: Wing Sweep at 25% MAC
- $S_{csw}$: Control surface area

In [155]:
def wing_weight(geom_def, wing_weight_parm, ss_areas):
    #Read JSON file
    with open(f"{geom_def}", "r") as file:
        geom = json.load(file)

    wing = geom["wing"]
    hstab = geom["hstab"]
    vstab = geom["vstab"]

    #Define Parameters
    W_dg = wing_weight_parm["W_dg"]
    N_z = wing_weight_parm["N_z"]
    S_w = float(wing["S_w"])
    A = float(wing["ar_w"])
    tc_rt = wing_weight_parm["tc_rt"]
    lam_w = float(wing["lamb_w"])
    Lam_25mac = np.deg2rad(wing_weight_parm["Lambda_25"])
    S_csw = float(ss_areas["Wing_Aileron_Area"]) + float(ss_areas["Wing_Slat_Area"]) + float(ss_areas["Wing_Flap_Area"])

    wing_weight_coeff = {
        "W_dg": W_dg,
        "N_z": N_z,
        "S_w": S_w,
        "A_w": A,
        "tc_rt": tc_rt,
        "lam_w": lam_w,
        "Lam_25mac_rad": Lam_25mac,
        "S_csw": S_csw
    }

    display(wing_weight_coeff)

    W_w = 0.0103 * (W_dg * N_z)**(0.5) * S_w**(0.622) * A**(0.785) * tc_rt * (1 + lam_w)**(0.05) * (np.cos(Lam_25mac))**(-1.0) * S_csw**(0.04)
    print(f"Wing Weight is: {W_w:.2f} lbs")

    #Conversion from lbs to slug
    M_w_slug = W_w / 32.174
    print(f"Wing Mass is: {W_w_slug:.2f} slugs")

    return M_w_slug
    

wing_weight_parm = {
    "W_dg": 49765,  #Design Gross Weight (raymer says 50% fuel, but this is full fuel)
    "N_z": 10.5,    #Design Ultimate Load Factor 
    "tc_rt": 4,  #Airfoil Thickness Percentage at Root
    "Lambda_25": 32, #Wing sweep at 35% MAC !! RECALCULATE THIS BASED ON COMPGEOM !!
}

geom_def = "airplane_geom.json"

M_w_slug = wing_weight(wing_weight_parm=wing_weight_parm, ss_areas=ss_areas, geom_def=geom_def)

{'W_dg': 49765,
 'N_z': 10.5,
 'S_w': 465.0,
 'A_w': 3.0,
 'tc_rt': 4,
 'lam_w': 0.27,
 'Lam_25mac_rad': np.float64(0.5585053606381855),
 'S_csw': 251.43}

Wing Weight is: 4791.11 lbs
Wing Mass is: 148.86 slugs


### Weights - HStab
Raymer Eq. 15.2 statistically estimates hstab weight:
$\begin{equation}
W_{HT}=3.316\bigg( 1 + \frac{F_w}{B_h} \bigg)^{-2.0}\bigg( \frac{W_{dg}N_z}{1000} \bigg)^{0.260}S_{ht}^{0.806}
\end{equation}$

Where: 
- $F_w$: Fuselage width at horizontal tail intersection (8.3ft)
- $B_h$: Horiz. tail span
- $W_{dg}$: Flight Design Gross Weight (typically 50-60% internal fuel)
- $N_z = 10.5$
- $S_{ht}$: Horiz. tail area

In [156]:
def hstab_weight(hstab_weight_parm, geom_def):
    #Read JSON file
    with open(f"{geom_def}", "r") as file:
        geom = json.load(file)

    wing = geom["wing"]
    hstab = geom["hstab"]
    vstab = geom["vstab"]

    #Define Parameters    
    F_w = hstab_weight_parm["F_w"]
    B_h = hstab["b_HT"]
    W_dg = hstab_weight_parm["W_dg"]
    N_z = hstab_weight_parm["N_z"]
    S_ht = hstab["S_HT"]

    hstab_weight_coeffs = {
        "F_w": F_w,
        "B_h": B_h,
        "W_dg": W_dg,
        "N_z": N_z,
        "S_ht": S_ht
    }
    display(hstab_weight_coeffs)

    W_HT = 3.316 * (1 + F_w / B_h)**(-2.0) * ((W_dg * N_z) / 1000)**(0.260) * S_ht**(0.806)
    print(f"Hstab Weight is: {W_HT:.2f} lbs")

    #Conversion from lbs to slug
    M_HT_slug = W_HT / 32.174
    print(f"Hstab Mass is: {M_HT_slug:.2f} slugs")

    return M_HT_slug



hstab_weight_parm = {
    "F_w": 8.3,
    "N_z": 10.5,
    "W_dg": 49765
}

M_HT_slug = hstab_weight(hstab_weight_parm=hstab_weight_parm, geom_def=geom_def)

{'F_w': 8.3,
 'B_h': 20.976942704647882,
 'W_dg': 49765,
 'N_z': 10.5,
 'S_ht': 146.67737507802667}

Hstab Weight is: 482.89 lbs
Hstab Mass is: 15.01 slugs


### Weights - VStab
Raymer Eq. 15.3 statistically estimates hstab weight:
$\begin{equation}
W_{VT} = 0.452K_{rht}(1 + H_t/H_v)^{0.5}(W_{dg}N_z)^{0.488}S_{vt}^{0.718}M^{0.341}\times L_{t}^{-1.0}(1+S_r/S_{vt})^{0.348}A_{vt}^{0.233} \times (1+\lambda)^{0.25}(\text{cos}\Lambda_{vt})^{-0.323}
\end{equation}$

Where: 
- $K_{rht} = 1.0$
- $H_t/H_v = 0$
- $W_{dg}$: Flight Design Gross Weight (typically 50-60% internal fuel)
- $N_z = 10.5$
- $S_{vt}$: Vertical Tail Area
- $M$: Design max mach number
- $L_t$: Tail length; wing quarter-MAC to tail quarter-MAC
- $S_r$: Rudder Area
- $A_{vt}$: Aspect Ratio, vertical tail
- $\lambda$: Taper Ratio, vertical tail
- $\Lambda_{vt}$: Sweep at 25% MAC, vertical tail

In [157]:
def vstab_weight(geom_def, vstab_weight_parm, ss_areas):
    #Read JSON file
    with open(f"{geom_def}", "r") as file:
        geom = json.load(file)

    wing = geom["wing"]
    hstab = geom["hstab"]
    vstab = geom["vstab"]

    #Define Parameters
    W_dg = vstab_weight_parm["W_dg"]
    N_z = vstab_weight_parm["N_z"]
    S_vt = vstab["S_VT"]
    M = vstab_weight_parm["M"]
    L_t = vstab["L_VT"]
    S_r = float(ss_areas["VStab_Rudder_Area"])
    A_VT = vstab["AR_VT"]
    lam_VT = vstab["lam_VT"]
    Lam_25mac = np.rad2deg(vstab_weight_parm["Lam_mac25"])

    vstab_weight_coeffs = {
        "W_dg": W_dg,
        "N_z": N_z,
        "S_vt": S_vt,
        "M": M,
        "L_t": L_t,
        "S_r": S_r,
        "A_VT": A_VT,
        "lam_VT": lam_VT,
        "Lam_25mac_rad": Lam_25mac
    }
    display(vstab_weight_coeffs)

    W_VT = 0.452 * (1 + 0)**(0.5) * (W_dg * N_z)**(0.488) * S_vt**(0.718) * M**(0.341) * L_t**(-1) * (1 + S_r / S_vt)**(0.348) * A_VT**(0.223) * (1 + lam_VT)**(0.25) * (np.cos(Lam_25mac))**(-0.323)
    print(f"Vertical Stabilizer Weight is: {W_VT:.2f} lbf")

    #Conversion from lbs to slug
    M_VT_slug = W_VT / 32.174
    print(f"Vstab Mass is: {M_VT_slug:.2f} slugs")

    return M_VT_slug


vstab_weight_parm = {
    "W_dg": 49765,
    "N_z": 10.5,
    "M": 1.6,
    "Lam_mac25": 35
}

M_VT_slug = vstab_weight(geom_def=geom_def, vstab_weight_parm=vstab_weight_parm, ss_areas=ss_areas)

{'W_dg': 49765,
 'N_z': 10.5,
 'S_vt': 102.91917001004019,
 'M': 1.6,
 'L_t': 13.5,
 'S_r': 53.331,
 'A_VT': 1.35,
 'lam_VT': 0.4,
 'Lam_25mac_rad': np.float64(2005.3522829578812)}

Vertical Stabilizer Weight is: 1118.08 lbf
Vstab Mass is: 34.75 slugs


### Assign Wing, HStab, VStab weights

In [158]:
def assign_mass(plane, surf_vols, W_w, W_HT, W_VT):
    tot_surf_mass = W_w + W_HT + W_VT
    print(f"Total Flying Surfaces Mass: {tot_surf_mass:.2f} slugs")

    #Clear and load model
    vsp.ClearVSPModel()
    vsp.ReadVSPFile(plane)

    wing_id = vsp.FindGeomsWithName("Main_Wing")[0]
    hstab_id = vsp.FindGeomsWithName("HStab")[0]
    vstab_id = vsp.FindGeomsWithName("VStab")[0]

    #Assign Density of Main Wing
    vsp.SetParmVal(wing_id, "Density", "Mass_Props", W_w / surf_vols["Wing_Vol_tot"])
    vsp.SetParmVal(wing_id, "Shell_Flag", "Mass_Props", 0.0)

    #Assign Density of Hstab
    vsp.SetParmVal(hstab_id, "Density", "Mass_Props", W_HT / surf_vols["Hstab_Vol_tot"])
    vsp.SetParmVal(hstab_id, "Shell_Flag", "Mass_Props", 0.0)

    #Assign Density of Vstab
    vsp.SetParmVal(vstab_id, "Density", "Mass_Props", W_VT / surf_vols["VStab_Vol_tot"])
    vsp.SetParmVal(vstab_id, "Shell_Flag", "Mass_Props", 0.0)

    vsp.Update()
    vsp.WriteVSPFile(plane)

    
assign_mass(plane=f"{vsp_filename}.vsp3", surf_vols=surf_vols, W_w=M_w_slug, W_HT=M_HT_slug, W_VT=M_VT_slug)

Total Flying Surfaces Mass: 198.67 slugs
